# 🎬 Kuku AI Microdrama Promo Finder & Growth Studio
**Kuku — AI Generalist Assignment (Problem 1: Find the Promos)**  
**Candidate:** Kumar Aryan (Roll: 23065060, Dept: Civil Engineering)  

This notebook is an interactive, runnable prototype that:
1. Parses microdrama screenplays with precise start/end timestamps snapped to camera and dialogue cuts.
2. Detects high-arousal **3-second hooks** (slaps, altar betrayals, weapon draws, high-conflict confrontations).
3. Scores **cliffhangers** (unresolved questions, facial defiance hardening, sound FX cliffhanger swells).
4. Assembles both single-stretch (20–60s) and **multi-segment contrast promos** (the classic ReelShort *Problem ➔ Duke Reveal* arc).
5. Executes an **Honest Quality Audit** exposing edge cases (pronoun ambiguity, reaction clipping) with human-in-the-loop remediation.
6. Schedules a **3-Phase Growth Rollout** for TikTok, Reels, and YouTube Shorts.

In [ ]:
# @title 1. Setup & Dependencies
import re
import os
import json
import pandas as pd
from typing import List, Dict, Any, Optional

print('Environment ready!')

In [ ]:
# @title 2. Core Screenplay Parser Engine
def parse_timestamp(ts_str: str) -> float:
    ts_str = ts_str.strip().strip('()')
    parts = ts_str.split(':')
    return float(parts[0]) * 60.0 + float(parts[1]) if len(parts) == 2 else float(parts[0])

def format_timestamp(seconds: float) -> str:
    mins = int(seconds // 60)
    secs = seconds % 60
    return f'{mins:02d}:{secs:05.2f}'

class ScriptBlock:
    def __init__(self, block_type, speaker, tone, start_time, end_time, content):
        self.block_type = block_type
        self.speaker = speaker
        self.tone = tone
        self.start_time = start_time
        self.end_time = end_time
        self.start_str = format_timestamp(start_time)
        self.end_str = format_timestamp(end_time)
        self.content = content

class Episode:
    def __init__(self, ep_num, title, logline, blocks):
        self.episode_number = ep_num
        self.episode_title = title
        self.logline = logline
        self.all_blocks = blocks

def parse_screenplay_text(text: str, filename: str = 'episode.txt') -> Episode:
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    ep_num = 1
    ep_match = re.search(r'EPISODE NUMBER:\s*(?:Episode\s*)?(\d+)', text, re.IGNORECASE)
    if ep_match:
        ep_num = int(ep_match.group(1))
    logline = ''
    log_match = re.search(r'EPISODE LOGLINE:\s*(.*?)(?=\n[A-Z\s]+:|\n---)', text, re.DOTALL | re.IGNORECASE)
    if log_match:
        logline = log_match.group(1).strip()
    
    block_pattern = re.compile(
        r'\*\*([^*]+?):?\*\*:?\s*\nStart:\s*\(([0-9:.]+)\)\s*\|\s*End:\s*\(([0-9:.]+)\)\s*\n(.*?)(?=(?:\n\*\*[^*]+?:?\*\*:?|\n--- SCRIPT END ---|\Z))',
        re.DOTALL
    )
    blocks = []
    for b_match in block_pattern.finditer(text):
        raw_tag = b_match.group(1).strip().rstrip(':')
        b_start = parse_timestamp(b_match.group(2).strip())
        b_end = parse_timestamp(b_match.group(3).strip())
        raw_content = b_match.group(4).strip()
        tag_upper = raw_tag.upper()
        if tag_upper == 'CAMERA':
            b_type, speaker, tone = 'CAMERA', None, None
        elif tag_upper in ('ACTION', 'ACTIONS'):
            b_type, speaker, tone = 'ACTION', None, None
        elif any(k in tag_upper for k in ('SOUND', 'SFX', 'MUSIC')):
            b_type, speaker, tone = 'SOUND_FX', None, None
        else:
            b_type = 'DIALOGUE'
            sp_match = re.match(r'^([^(]+)(?:\((.*)\))?$', raw_tag)
            speaker = sp_match.group(1).strip() if sp_match else raw_tag
            tone = sp_match.group(2).strip() if sp_match and sp_match.group(2) else None
            if raw_content.startswith('"') and raw_content.endswith('"'):
                raw_content = raw_content[1:-1].strip()
        blocks.append(ScriptBlock(b_type, speaker, tone, b_start, b_end, raw_content))
    return Episode(ep_num, f'Episode {ep_num}', logline, blocks)

print('Screenplay parser engine defined.')

In [ ]:
# @title 3. Feature Extraction, Hook & Cliffhanger Scoring
ACTION_HOOK_KEYWORDS = {'slap': 4.0, 'strike': 3.5, 'hit': 3.5, 'shove': 3.0, 'mud': 3.5, 'blood': 4.0, 'dagger': 3.5, 'sword': 3.5, 'kneel': 3.0, 'grab': 3.0}
DIALOGUE_HOOK_KEYWORDS = {'marrying': 3.5, 'betrayal': 4.0, 'disowned': 4.0, 'witchcraft': 4.0, 'liar': 3.5, 'worthless': 3.5, 'duke': 3.5, 'kill': 4.0, 'soldier': 2.5}
INTENSE_TONES = {'cold': 2.0, 'furious': 2.5, 'cruel': 2.5, 'smug': 2.5, 'gloating': 2.5, 'arrogant': 2.5, 'mocking': 2.5, 'screaming': 3.0, 'defiant': 2.5}
KEY_CHARACTERS = {'juliana', 'gerald', 'clarissa', 'alexander', 'duke', 'matriarch', 'catherine'}
AMBIGUOUS_OPENERS = {'he', 'she', 'they', 'it', 'him', 'her', 'them', 'his', 'their'}

def score_promo_candidate(blocks, ep_num):
    start_t = blocks[0].start_time
    end_t = blocks[-1].end_time
    dur = end_t - start_t
    
    # 1. Hook Score (First 3.5s)
    hook_blocks = [b for b in blocks if b.start_time <= start_t + 3.5]
    if not hook_blocks: hook_blocks = [blocks[0]]
    hook_score = 2.0
    hook_desc = ''
    hook_type = 'Shock Dialogue'
    for b in hook_blocks:
        low = b.content.lower()
        if b.block_type == 'ACTION':
            for kw, w in ACTION_HOOK_KEYWORDS.items():
                if kw in low: hook_score += w
            if not hook_desc: hook_desc = b.content[:80]
            hook_type = 'Action Disruption'
        elif b.block_type == 'DIALOGUE':
            for kw, w in DIALOGUE_HOOK_KEYWORDS.items():
                if kw in low: hook_score += w
            if '!' in b.content: hook_score += 1.5
            if '?' in b.content: hook_score += 1.0
            if b.tone and any(t in b.tone.lower() for t in INTENSE_TONES):
                hook_score += 2.0
            if not hook_desc: hook_desc = f'{b.speaker}: "{b.content[:70]}"'
    if hook_blocks[0].block_type == 'DIALOGUE':
        first_w = hook_blocks[0].content.strip().split()[0].lower().strip(""""'.,;:""")
        if first_w in AMBIGUOUS_OPENERS:
            hook_score -= 1.5
    hook_score = min(10.0, max(1.0, hook_score))
    
    # 2. Cliffhanger Score (Last 4.0s)
    tail_blocks = [b for b in blocks if b.end_time >= end_t - 4.0]
    if not tail_blocks: tail_blocks = [blocks[-1]]
    cliff_score = 2.5
    cliff_desc = ''
    for b in tail_blocks:
        low = b.content.lower()
        if b.block_type == 'SOUND_FX' and any(k in low for k in ['cliffhanger', 'drone', 'swell', 'sting']):
            cliff_score += 3.5
            cliff_desc = b.content[:80]
        elif b.block_type == 'DIALOGUE' and '?' in b.content:
            cliff_score += 3.0
            if not cliff_desc: cliff_desc = f'{b.speaker}: "{b.content[:70]}"'
    cliff_score = min(10.0, max(1.0, cliff_score))
    if not cliff_desc: cliff_desc = 'Dramatic cut at unresolved tension'
    
    # 3. Standalone Comprehensibility
    total_txt = ' '.join(b.content for b in blocks).lower()
    standalone_score = 4.0
    if any(c in total_txt for c in KEY_CHARACTERS): standalone_score += 3.0
    if any(k in total_txt for k in ['wedding', 'altar', 'disown', 'duke', 'soldier']): standalone_score += 3.0
    standalone_score = min(10.0, standalone_score)
    
    # 4. Pacing & Duration (25-45s sweet spot)
    dur_score = 10.0 if 25.0 <= dur <= 45.0 else (8.0 if 20.0 <= dur <= 60.0 else 4.0)
    
    composite = (hook_score * 3.5) + (cliff_score * 3.0) + (standalone_score * 2.0) + (dur_score * 1.5)
    return {
        'episode': ep_num,
        'duration': round(dur, 2),
        'start_time': blocks[0].start_str,
        'end_time': blocks[-1].end_str,
        'hook_score': round(hook_score, 1),
        'hook_type': hook_type,
        'hook_desc': hook_desc,
        'cliff_score': round(cliff_score, 1),
        'cliff_desc': cliff_desc,
        'composite_score': round(composite, 2)
    }

print('Feature extraction & scoring ready.')

In [ ]:
# @title 4. Run Discovery on Sample Screenplays
# Embedded sample screenplays from Episode 1 and Episode 29
sample_ep1 = """EPISODE NUMBER: Episode 1\nEPISODE LOGLINE: Lady Juliana arrives at the cathedral in her late mother's bridal gown, only to suffer the ultimate betrayal when her fiancé Gerald abandons her at the altar for her cruel sister Clarissa with her family's full blessing.\nCONTINUITY NOTES: First episode of the series. Establishes Juliana's humiliation and abandonment by her betrothed, Gerald, and her noble family in favor of her sister, Clarissa.\n\n--- SCRIPT START ---\n\nSCENE 01\nScene heading: INT. CATHEDRAL – DAY\nScene start: 00:00.00\nScene end: 02:18.00\n\nActions and dialogue:\n\n**CAMERA:**\nStart: (00:00.00) | End: (00:04.80)\nSymmetrical framing on the heavy church doors as they slowly part, pushing in slightly as Juliana stands framed in the light, dressed in an elegant white silk gown.\n\n**ACTION:**\nStart: (00:01.00) | End: (00:04.80)\nJuliana pauses at the threshold, her expression bright with joyful anticipation, wearing her mother's vintage gown and an oval pendant necklace.\n\n**CAMERA:**\nStart: (00:04.80) | End: (00:06.80)\nHigh wide tracking shot following behind Juliana as she steps into the grand cathedral nave, facing the distant altar flanked by seated guests.\n\n**CAMERA:**\nStart: (00:06.80) | End: (00:09.80)\nClose-up on Juliana's face as her warm smile begins to falter into confusion.\n\n**ACTION:**\nStart: (00:07.20) | End: (00:09.80)\nJuliana's eyes widen slightly in bewilderment as she notices something amiss at the altar.\n\n**CAMERA:**\nStart: (00:09.80) | End: (00:11.00)\nMedium profile shot at the altar showing Gerald in his dark tailcoat holding the gloved hand of the Lady in Gold Gown, with Juliana visible walking down the distant aisle.\n\n**CAMERA:**\nStart: (00:11.00) | End: (00:13.20)\nMedium shot as Juliana stops midway down the aisle, confronted by a haughty matriarch in black velvet stepping into the foreground.\n\n**MATRIARCH (Cold, Commanding):**\nStart: (00:11.30) | End: (00:13.10)\n\"You should turn around, Juliana.\"\n\n**LADY JULIANA (Confused, Breathless):**\nStart: (00:13.50) | End: (00:14.30)\n\"Why?\"\n\n**CAMERA:**\nStart: (00:14.30) | End: (00:17.80)\nOver-the-shoulder medium close-up past the Matriarch to Juliana, looking past her toward the altar.\n\n**LADY JULIANA (Desperate, Unbelieving):**\nStart: (00:15.50) | End: (00:17.50)\n\"Gerald is waiting for me.\"\n\n**CAMERA:**\nStart: (00:17.80) | End: (00:20.50)\nClose-up on the Matriarch wearing an ornate tiara and sapphire jewelry, gazing down with merciless disdain.\n\n**MATRIARCH (Blunt, Unforgiving):**\nStart: (00:18"""
sample_ep29 = """EPISODE NUMBER: Episode 29\nEPISODE LOGLINE: Thrown into the castle's subterranean wet cells as Catherine threatens to burn her mother's herbarium and frame her for witchcraft, Juliana breaks down in isolation before swearing that they will never take Alexander from her.\nCONTINUITY NOTES: Follows directly from Episode 28 where Rupert ordered the guards to drag Juliana down to the wet cells while Catherine seized her mother's herbarium.\n\n--- SCRIPT START ---\n\nSCENE 01\nScene heading: INT. CASTLE DUNGEON STAIRWELL – NIGHT\nScene start: 00:00.00\nScene end: 00:20.10\n\nActions and dialogue:\n\n**CAMERA:**\nStart: (00:00.00) | End: (00:02.80)\nHigh angle tracking shot descending a spiral stone staircase illuminated by flickering wall torches and damp, glistening masonry.\n\n**SOUND FX:**\nStart: (00:00.00) | End: (00:03.50)\nDripping water echoes hollowly against stone steps alongside the shuffling scuffle of boots.\n\n**CAMERA:**\nStart: (00:02.80) | End: (00:05.10)\nTracking shot facing two stern uniform guards hauling a struggling Juliana down the dark corridor by her arms.\n\n**ACTION:**\nStart: (00:02.90) | End: (00:05.10)\nJuliana resists furiously, her feet stumbling on the wet flagstones as the guards forcibly propel her forward.\n\n**CAMERA:**\nStart: (00:05.10) | End: (00:07.40)\nMedium low-angle shot of Catherine standing regally under an arched stone doorway, clutching the leather-bound herbarium.\n\n**CAMERA:**\nStart: (00:07.40) | End: (00:11.70)\nTight close-up of Juliana's tearful face as she reaches an arm out toward Catherine in anguish.\n\n**JULIANA (Desperate, Weeping):**\nStart: (00:07.40) | End: (00:11.70)\n\"Catherine! Please! That book is my mother's only legacy!\"\n\n**CAMERA:**\nStart: (00:11.70) | End: (00:13.20)\nMedium shot from behind Catherine as she stands motionless, watching Juliana being dragged away.\n\n**CAMERA:**\nStart: (00:13.20) | End: (00:17.30)\nClose-up of Juliana turning her head back toward Catherine, screaming through tears.\n\n**JULIANA (Pleading, Agonized):**\nStart: (00:13.20) | End: (00:17.30)\n\"It contains years of her work! Her knowledge! Everything she left me!\"\n\n**CAMERA:**\nStart: (00:17.30) | End: (00:20.10)\nFull shot of Catherine turning gracefully on the dungeon steps, holding the book loosely at her side.\n\nSCENE 02\nScene heading: INT. DUNGEON CORRIDOR – CONTINUOUS\nScene start: 00:20.10\nScene end: 01:23.00\n\nActions and dialogue:\n\n**CAMERA:**\nStart: (00:20.10) | End: (00:26.80)\nMedium close-up pushing in on Catherine as she presents the worn botanical"""

episodes = [
    parse_screenplay_text(sample_ep1, 'episode_001.txt'),
    parse_screenplay_text(sample_ep29, 'episode_029.txt'),
]

all_promos = []
for ep in episodes:
    n = len(ep.all_blocks)
    ep_cands = []
    for i in range(n):
        st_t = ep.all_blocks[i].start_time
        for j in range(i, n):
            en_t = ep.all_blocks[j].end_time
            dur = en_t - st_t
            if 20.0 <= dur <= 60.0:
                cand_res = score_promo_candidate(ep.all_blocks[i:j+1], ep.episode_number)
                ep_cands.append(cand_res)
            elif dur > 60.0:
                break
    ep_cands.sort(key=lambda x: x['composite_score'], reverse=True)
    if ep_cands:
        all_promos.append(ep_cands[0])

df = pd.DataFrame(all_promos)
print('Discovered Top Promos:')
display(df[['episode', 'start_time', 'end_time', 'duration', 'composite_score', 'hook_desc', 'cliff_desc']])

### 🏆 Growth Rollout Strategy & Rationale
- **Phase 1 (Post Days 1–2):** Shock & Outrage Ads (Ep 29 Witchcraft accusation, Ep 1 Altar abandonment) to flood top-of-funnel TikTok with highest 3s thumbstop.
- **Phase 2 (Post Days 3–5):** Secret Duke Reveal & Catharsis (Broken soldier revealed as war hero Duke) to drive app installs and episode unlocks.
- **Phase 3 (Post Day 6+):** Mid-Series Intrigue & Retargeting to convert users past paywalls.